# 05e – Histogram gradient boosting (Poisson), getuned

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Het snelle model uit 05a tunen, en onderzoeken hoeveel "recente geschiedenis" (lag-kenmerken) zou opleveren. |
| **Input** | `data/model_table.parquet` |
| **Output** | `models/metrics/hist_gradient_boosting.json`, `models/metrics/hgb_with_lags.json`, `models/predictions/*.parquet`, `models/best_params.json` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

## Waarom dit model?
**Gradient boosting** bouwt bomen **na elkaar**: elke nieuwe (kleine) boom leert de fouten van alle vorige samen. Het is op tabeldata meestal het sterkste model.
- De **histogram**-variant van scikit-learn deelt elke numerieke kolom eerst in 255 bakjes in. Dat maakt het snel op honderdduizenden rijen (vergelijkbaar met LightGBM).
- **`loss="poisson"`** optimaliseert rechtstreeks de Poisson-deviance: het juiste verlies voor tellingen, en voorspellingen zijn altijd > 0.
- **Zone als echte categorie** (`categorical_features`): een split kan elke groep zones afzonderen, niet alleen "zone ≤ k".
- Het draait met alleen scikit-learn, dus ook op de server zonder extra bibliotheken. Daarom is dit de kandidaat voor de deployment.

## 0. Imports en data

In [1]:
import json

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV

import citibike as cb

table = cb.load_model_table()
X_train, y_train, g_train, X_test, y_test = cb.train_test(table)
ZONE = [cb.FEATURES.index("zone")]

## 1. Randomized search
| Hyperparameter | Effect |
|---|---|
| `learning_rate` | hoe groot elke correctiestap is: klein = voorzichtiger, vraagt meer bomen |
| `max_iter` | aantal bomen |
| `max_leaf_nodes` | complexiteit van elke boom (hoeveel interacties) |
| `min_samples_leaf` | minimum aantal uren per blad: tegen het volgen van ruis |
| `l2_regularization` | straf op grote bladwaarden |

25 combinaties × 5 folds (op kalenderweek gegroepeerd), gekozen op MAE.

In [2]:
search = RandomizedSearchCV(
    HistGradientBoostingRegressor(loss="poisson", categorical_features=ZONE, random_state=cb.SEED),
    {"learning_rate": loguniform(0.02, 0.3), "max_iter": randint(200, 1200),
     "max_leaf_nodes": randint(15, 127), "min_samples_leaf": randint(10, 200),
     "l2_regularization": loguniform(1e-3, 10)},
    n_iter=25, cv=GroupKFold(5), scoring="neg_mean_absolute_error", random_state=cb.SEED, n_jobs=1)
search.fit(X_train, y_train, groups=g_train)
results = (pd.DataFrame(search.cv_results_["params"])
             .assign(cv_mae=-search.cv_results_["mean_test_score"], std_over_folds=search.cv_results_["std_test_score"],
                     fit_seconds=search.cv_results_["mean_fit_time"])
             .sort_values("cv_mae"))
display(results.head(8).round(3))
print("Slechtste combinaties:")
results.tail(4).round(3)

,l2_regularization,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,cv_mae,std_over_folds,fit_seconds
18,0.015,0.031,1056,85,18,24.902,1.279,16.736
17,0.027,0.043,579,123,50,25.086,1.441,11.246
10,0.008,0.038,1155,85,181,25.136,1.540,15.628
1,0.244,0.067,414,89,97,25.149,1.434,6.301
13,0.187,0.022,601,104,53,25.192,1.347,11.141
19,1.227,0.034,671,77,148,25.192,1.386,9.536
11,0.096,0.022,405,95,173,25.235,1.258,6.849
6,0.029,0.069,886,76,60,25.278,1.542,11.826


Slechtste combinaties:


,l2_regularization,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,cv_mae,std_over_folds,fit_seconds
14,5.728,0.226,469,109,57,26.638,1.551,7.642
2,0.022,0.029,330,36,62,26.856,1.002,2.976
0,0.031,0.263,306,86,198,26.882,1.657,4.649
24,0.077,0.028,661,17,10,27.561,0.880,3.750


**Interpretatie:**
- De beste combinatie haalt CV-MAE **24,9**. De top 8 ligt tussen 24,9 en 25,3, terwijl de spreiding over de folds ± 1,3 tot 1,5 is: binnen de top maakt de exacte keuze weinig uit.
- Wat de beste combinaties delen: een **kleine learning rate** (0,02 tot 0,07) met **grote bomen** (75 tot 125 bladeren). Veel kleine correctiestappen met bomen die genoeg interacties kunnen vatten (zone × uur × weekend × weer).
- De slechtste combinaties hebben ofwel een **grote learning rate** (0,23 tot 0,26: te grote stappen, het model schiet voorbij), ofwel **kleine bomen** (17 tot 36 bladeren: te weinig ruimte voor de interacties).

## 2. Beste model: cross-validatie en eindtest

In [3]:
params = {k: (round(float(v), 6) if isinstance(v, float) else int(v)) for k, v in search.best_params_.items()}
best = HistGradientBoostingRegressor(loss="poisson", categorical_features=ZONE, random_state=cb.SEED, **params)
cv = cb.cv_scores(best, X_train, y_train, g_train)
best.fit(X_train, y_train)
pred = best.predict(X_test)
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "05e_hist_gradient_boosting", "model": "HistGradientBoosting (Poisson, getuned)",
           "params": params, **cv, **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("hist_gradient_boosting", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred))
quick = json.loads((cb.METRICS_DIR / "quick_hgb.json").read_text())
pd.DataFrame({"standaard (05a)": quick, "getuned": metrics}).loc[["cv_mae_mean", "test_mae", "test_rmse", "test_r2", "test_poisson_deviance"]]

,standaard (05a),getuned
cv_mae_mean,27.5663,24.8879
test_mae,30.8826,29.0454
test_rmse,57.1495,56.3468
test_r2,0.9029,0.9056
test_poisson_deviance,13.3123,12.5921


**Interpretatie:** tunen levert een duidelijke maar bescheiden winst op: CV-MAE van 27,6 naar 24,9, en dat in **alle 5 folds**. Op de testset: van 30,9 naar 29,0 (R² 0,903 naar 0,906). De grootste stap zat dus niet in het tunen, maar in de keuze van het model en de kenmerken (baseline 59 naar 31).

Opvallend: de testfout (29,0) is duidelijk hoger dan de CV-fout (24,9). De testweken zijn blijkbaar moeilijker dan een gemiddelde week (feestdagen zoals Thanksgiving en Kerstmis vallen erin). In `06` zoeken we uit waar die fouten zitten.

## 3. Experiment: wat als het model de recente vraag kende? (lag-kenmerken)
In AWS-hoofdstuk 4 zagen we dat tijdreeksen **autocorrelatie** hebben: het aantal ritten nu lijkt op dat van gisteren op hetzelfde uur en van vorige week op hetzelfde uur. We voegen twee **lags** toe:
- `lag_24h`: vertrekken in dezelfde zone 24 uur eerder;
- `lag_168h`: vertrekken in dezelfde zone 168 uur (een week) eerder.

**Waarom zit dit niet in het gedeployde model?** Bij een voorspelling voor morgen of volgende week kent de gebruiker van onze webpagina die getallen niet: daarvoor zou de API een live koppeling met de Citi Bike-data nodig hebben (bv. de officiële GBFS-feed). Dit experiment toont hoeveel dat zou opleveren, en dus of zo'n koppeling de moeite waard is.

Let op: de eerste week van januari heeft geen lag_168h (NaN); HistGradientBoosting kan daar zelf mee om.

In [4]:
lagged = table.sort_values(["zone", "time"]).copy()
for hours in [24, 168]:
    lagged[f"lag_{hours}h"] = lagged.groupby("zone")["trips"].shift(hours)
lagged = lagged.loc[table.index]
lag_features = cb.FEATURES + ["lag_24h", "lag_168h"]
X_lag = pd.concat([cb.make_features(lagged), lagged[["lag_24h", "lag_168h"]]], axis=1)
is_test = (table["split"] == "test").to_numpy()

with_lags = HistGradientBoostingRegressor(loss="poisson", categorical_features=ZONE, random_state=cb.SEED, **params)
cv_lag = cb.cv_scores(with_lags, X_lag[~is_test], y_train, g_train)
with_lags.fit(X_lag[~is_test], y_train)
pred_lag = with_lags.predict(X_lag[is_test])
lag_metrics = cb.evaluate(y_test, pred_lag)
cb.save_results("hgb_with_lags", {"notebook": "05e_hist_gradient_boosting",
                                  "model": "HistGradientBoosting + lag 24h/168h (niet deploybaar)",
                                  "params": params, **cv_lag, **{f"test_{k}": v for k, v in lag_metrics.items()}},
                table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred_lag))
pd.DataFrame({"zonder lags": test_metrics, "met lags": lag_metrics})

,zonder lags,met lags
mae,29.0454,26.8966
rmse,56.3468,51.0475
r2,0.9056,0.9225
poisson_deviance,12.5921,10.3574


**Interpretatie:** het beeld is **niet eenduidig**, en dat is zelf een resultaat.
- Op de **testset** helpen de lags: MAE 29,0 naar 26,9, R² 0,906 naar 0,923.
- In de **cross-validatie** zijn ze gemiddeld juist **slechter** (27,5 tegenover 24,9) en winnen ze maar in 0 van de 5 folds.

Mogelijke verklaringen: de hyperparameters zijn afgestemd op het model zonder lags, en een lag van een uitzonderlijke dag (feestdag, storm) stuurt het model een week of dag later de verkeerde kant op. Op de testset met veel feestdagen kan "gisteren was het ook rustig" net wel helpen. Met deze gegevens kunnen we niet besluiten dat een live koppeling de voorspellingen betrouwbaar beter maakt. Voor de deployment verandert er dus niets: het model zonder lags is ook het enige dat werkt zonder live data.

## 4. Hyperparameters doorgeven aan de pipeline
We schrijven de getunede waarden naar `models/best_params.json`. Dat bestand leest `train.py`. Wordt het naar GitHub gepusht, dan hertraint de workflow `citi_bike_demand_retrain.yml` het model, test het tegen het huidige model in productie, en zet het een pull request klaar. De merge van die PR deployt het naar de VM.

Hier draaien we `train.py` ook één keer lokaal, zodat de API-map meteen het getunede model bevat.

In [5]:
import subprocess
import sys

(cb.MODEL_DIR / "best_params.json").write_text(json.dumps(
    {"source": "05e_hist_gradient_boosting (randomized search, 25 x 5 folds)", "params": params}, indent=2))
result = subprocess.run([sys.executable, "train.py"], capture_output=True, text=True)
print(result.stdout[-1500:], result.stderr[-1500:])

Data: 263,520 zone-uren (203,040 train / 60,480 test), 30 zones
Hold-out: {'mae': 29.0454, 'rmse': 56.3468, 'r2': 0.9056, 'poisson_deviance': 12.5921}
Huidig model: MAE 29.0454; nieuw: 29.0454 (grens 29.6263)
Bundle geschreven naar C:\Users\kobev\Documents\semester 1\3APPAI - Deep Learning\CloudAI-Challenge\deploy\citi_bike_demand\model.pkl (4.7 MB)
 


## Samenvatting
| Variant | CV-MAE | Test-MAE | Test-R² |
|---|---|---|---|
| standaard (05a) | 27,6 | 30,9 | 0,903 |
| **getuned** (`learning_rate=0,031`, `max_iter=1056`, `max_leaf_nodes=85`, `min_samples_leaf=18`, `l2=0,015`) | **24,9** | **29,0** | **0,906** |
| getuned + lags (niet deploybaar) | 27,5 | 26,9 | 0,923 |

- Het getunede model is het beste deploybare model van alle notebooks (zie `06` voor de volledige vergelijking).
- De hyperparameters staan in `models/best_params.json`; `train.py` heeft het definitieve model (getraind op alle uren van 2024) naar `deploy/citi_bike_demand/model.pkl` geschreven (4,7 MB). De kwaliteitspoort keurde het goed.